In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from cliffs_delta import cliffs_delta
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.dependency_extractor import DependencyExtractor

## To explore the PR databaset: use the `df` variable and `all_upgrades` to explore Dependabot dependency upgrades

In [2]:
pr_metrics_df = pd.DataFrame({})
for attr in ['star', 'commit', 'contributor', 'dep']:
    item_df = pd.read_csv(f"../dependabot-security/data/{attr}_sample_prs.csv")
    # item_df['Dependency_Type_dev_runtime_dev'] = item_df.apply(
    # lambda row: 1 if row['Dependency_Type_dev_runtime_runtime'] == 0 and row['Dependency_Type_dev_runtime_optional'] == 0 else row['Dependency_Type_dev_runtime_dev'], axis=1)
    # item_df['Dependency_Type_dev_runtime_dev'] = item_df.apply(
    #     lambda row: 0 if row['Dependency_Type_dev_runtime_runtime'] == 1 else row['Dependency_Type_dev_runtime_dev'], axis=1)
    # item_df['Dependency_Type_dev_runtime_dev'] = item_df.apply(
    #     lambda row: 0 if row['Dependency_Type_dev_runtime_optional'] == 1 else row['Dependency_Type_dev_runtime_dev'], axis=1)

    # item_df.to_csv(f"../dependabot-security/data/{attr}_sample_prs.csv", index=None)

    pr_metrics_df = pd.concat((pr_metrics_df, item_df))

model_data_df = pd.read_csv("./data/model_data.csv")
pr_metrics_df = pd.concat((pr_metrics_df, model_data_df))

pr_metrics_df.drop_duplicates(subset=['repo', 'id'], keep='last', inplace=True)
pr_metrics_df.reset_index(drop=True, inplace=True)
pr_metrics_df['repo_pr_id'] = pr_metrics_df['repo'].str.cat(pr_metrics_df['id'].astype(int).astype(str), '&SEP&')

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_58766/1804341199.py:3: DtypeWarning: Columns (5,16,17,18,19,20,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  item_df = pd.read_csv(f"../dependabot-security/data/{attr}_sample_prs.csv")
/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_58766/1804341199.py:3: DtypeWarning: Columns (5,16,17,18,19,20,33,36) have mixed types. Specify dtype option on import or set low_memory=False.
  item_df = pd.read_csv(f"../dependabot-security/data/{attr}_sample_prs.csv")
/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_58766/1804341199.py:15: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  model_data_df = pd.read_csv("./data/model_data.csv")


### Combine PRs across the four metrics

In [3]:
closed_prs_committedate_df = pd.read_csv("./data/closed_prs_committedate.csv")

In [4]:
input_files = [
    ("base_prs_star.csv", "Star"),
    ("base_prs_commit.csv", "Commit"),
    ("base_prs_contributor.csv", "Contributor"),
    ("base_prs_dep.csv", "Dependabot")
]

bots = ['dependabot', 'github-actions',
        'contentful-automation', 'mergify', 'kodiakhq']

def retrieve_library_name(title):
    try:
        lib, _, _, prefix = DependencyExtractor.extract(title)
        if prefix:
            lib = f"{prefix}/{lib}"
        return lib
    except Exception as ex:
        print(ex)
        return None


def fill_pr_category(row):
    state = row['state']
    category = row['pr_category']
    if row['pr_category'] == "Many changed package managers":
        return "Others"
    elif state == "CLOSED":
        return category
    elif state == "MERGED":
        return "Up-to-date"
    elif state == "OPEN":
        return "No activity"
    return category


def calc_merge_time(row):
    return (row['pr_merged_at'] - row['pr_created_at']).total_seconds() / (60) if row['state'] == "MERGED" else None


# def fill_closed_missing_prs(row):
#     found_commitdate = closed_prs_committedate_kv.get(row['repo_pr_id'], None)

#     if (row['state'] != 'CLOSED') or (row['pr_category'] not in ['Up-to-date', 'Others', 'Unknown error', 'Git error']) or (found_commitdate is None) or (type(found_commitdate) == float):
#         return row

#     row['pr_category'] = "Up-to-date"
#     row['pr_merged_at'] = found_commitdate
#     return row


def process_df():
    df = pd.DataFrame({})
    for file, metric in input_files:
        item_df = pd.read_csv(f"./data/{file}")
        item_df['metric'] = metric
        df = pd.concat((df, item_df))

    df['pr_created_at'] = pd.to_datetime(df['pr_created_at'])
    df['pr_closed_at'] = pd.to_datetime(df['pr_closed_at'])
    df['pr_merged_at'] = pd.to_datetime(df['pr_merged_at'])
    df['repo_created_at'] = pd.to_datetime(df['repo_created_at'])
    df['repo_updated_at'] = pd.to_datetime(df['repo_updated_at'])

    df["repo_last_committed_date"] = pd.to_datetime(
        df["repo_last_committed_date"])
    df["repo_last_update"] = (datetime.now(
        timezone.utc) - df['repo_last_committed_date']).dt.total_seconds() / (3600 * 24)
    df['repo_age'] = (df['repo_last_committed_date'] -
                      df['repo_created_at']).dt.days

    df["pr_category"] = df.apply(fill_pr_category, axis=1)

    df.drop_duplicates(subset=['repo', 'id'], inplace=True)
    all_prs_len = len(df)

    df['repo_pr_id'] = df['repo'].str.cat(df['id'].astype(str), "&SEP&")
    df.loc[:, 'lib_name'] = df['title'].map(retrieve_library_name)
    df['repo_lib'] = df['repo'].str.cat(df['lib_name'].astype(str), "-")

    excluded_pr_categories = [
        'Non parsable',
        'Unchanged package manager',
        'No package manager',
        # 'Many changed package managers',
        'Unknown error',
        'Git error'
    ]

    # print("Number of all studied Dependabot PRs: {} for {} projects".format(
        # all_prs_len, df['repo'].nunique()))

    df['merge_time'] = df.apply(calc_merge_time, axis=1)

    all_dfs = df.copy()

    df = df[~df['pr_category'].isin(excluded_pr_categories)]
    df = df[(~df['merged_by'].isin(bots)) | (
        (df['merged_by'].isin(bots)) & (df['merge_time'] > 1))]

    df = df[df['mentionable_users_count'] >= 5]

    df['changed_files'] = df['changed_files'].map(eval)

    targets = ["yarn.lock", "package.json", "package-lock.json"]
    mask = df["changed_files"].apply(
        lambda files: any(f in targets for f in files) if isinstance(
            files, list) else False
    )
    df = df[mask]
    clean_prs_len = len(df)


    excluded_repos = ['choyiny/cscc09.com']

    repo_pr_count_df = df.groupby('repo').count().reset_index(level=0)
    repo_pr_less_than_5 = repo_pr_count_df.loc[repo_pr_count_df['id'] < 5, 'repo'].tolist()
    excluded_repos.extend(repo_pr_less_than_5)

    df = df[~df['repo'].isin(excluded_repos)]

    print("Number of clean studied Dependabot PRs: {} for {} projects.".format(
        clean_prs_len, df['repo'].nunique()))

    return all_dfs, df

all_dfs, df = process_df()

Number of clean studied Dependabot PRs: 92630 for 700 projects.


### Prepare superseding upgrades

In [5]:
def calc_delay_time(row):
    if pd.notna(row['last_pr_merged_at']):
        last_pr_time = row['last_pr_merged_at']
    elif pd.notna(row['last_pr_closed_at']):
        last_pr_time = row['last_pr_closed_at']
    else:
        last_pr_time = row['last_pr_created_at']
    return (last_pr_time - row['pr_created_at']).total_seconds() / (60)


# Function to count continuous superseded PRs and track changes
def count_continuous_superseded_sequences(prs):
    sequences = []
    current_sequence = []
    prior_superseded_prs = []
    for _, row in prs.iterrows():
        if row['pr_category'] == 'Superseded':
            current_sequence.append(row)
            prior_superseded_prs.append(row['id'])
        else:
            if current_sequence:
                # Store the pr_id and lib_name of the last PR in the sequence
                sequences.append({
                    'state': current_sequence[0]['state'],
                    'count': len(current_sequence),
                    'pr_id': current_sequence[0]['id'],
                    'pr_title': current_sequence[0]['title'],
                    'dependabot_exists': current_sequence[0]['dependabot_exists'],
                    'star_label': row['star_label'],
                    'commit_label': row['commit_label'],
                    'contributor_label': row['contributor_label'],
                    'dep_pr_label': row['dep_pr_label'],
                    'pr_created_at': current_sequence[0]['pr_created_at'],
                    'pr_closed_at': current_sequence[0]['pr_closed_at'],
                    'last_pr_id': row['id'],
                    'lib_name': current_sequence[-1]['lib_name'],
                    'last_pr_created_at': row['pr_created_at'],
                    'last_pr_closed_at': row['pr_closed_at'],
                    'last_pr_merged_at': row['pr_merged_at'],
                    'last_title': row['title'],
                    'last_pr_state': row['state'],
                    'last_category': row['pr_category'],
                    'last_merged_by': row['merged_by'],
                    'metric': row['metric'],
                    'prior_superseded_prs': prior_superseded_prs
                })
                current_sequence = []
                prior_superseded_prs = []
    # Check if the last sequence ended with a superseded PR
    if current_sequence:
        sequences.append({
            'count': len(current_sequence),
            'pr_id': current_sequence[0]['id'],
            'pr_title': current_sequence[0]['title'],
            'pr_created_at': current_sequence[0]['pr_created_at'],
            'pr_closed_at': current_sequence[0]['pr_closed_at'],
            'state': current_sequence[0]['state'],
            'dependabot_exists': current_sequence[0]['dependabot_exists'],
            'last_pr_id': current_sequence[-1]['id'],
            'lib_name': current_sequence[-1]['lib_name'],
            'star_label': current_sequence[0]['star_label'],
            'commit_label': current_sequence[0]['commit_label'],
            'contributor_label': current_sequence[0]['contributor_label'],
            'dep_pr_label': current_sequence[0]['dep_pr_label'],
            'metric': current_sequence[0]['metric'],
            'prior_superseded_prs': [current_sequence[0]['id']],
            'last_pr_closed_at': None,
            'last_pr_created_at': None,
            'last_pr_merged_at': None,
            'last_title': None,
            'last_pr_state': None,
            'last_category': None,
            'last_merged_by': None
        })

    return sequences


df.sort_values(by=['repo', 'pr_created_at'], inplace=True)

# Group by repo and lib_name, and apply the function
result = df.groupby(['repo', 'lib_name']).apply(
    lambda x: count_continuous_superseded_sequences(x)).reset_index()

# Flatten the result to create a DataFrame with each sequence
flattened_result = []
for index, row in result.iterrows():
    repo = row['repo']
    lib_name = row['lib_name']
    sequences = row[0]
    # print(sequences)
    for seq in sequences:
        flattened_result.append({
            'repo': repo,
            'id': seq['pr_id'],
            'title': seq['pr_title'],
            'pr_created_at': seq['pr_created_at'],
            'pr_closed_at': seq['pr_closed_at'],
            'state': seq['state'],
            'dependabot_exists': seq['dependabot_exists'],
            'lib_name': lib_name,
            'star_label': seq['star_label'],
            'commit_label': seq['commit_label'],
            'contributor_label': seq['contributor_label'],
            'dep_pr_label': seq['dep_pr_label'],
            'continuous_superseded_count': seq['count'],
            'last_pr_id': seq['last_pr_id'],
            'last_pr_closed_at': seq['last_pr_closed_at'],
            'last_pr_created_at': seq['last_pr_created_at'],
            'last_pr_merged_at': seq['last_pr_merged_at'] if pd.notna(seq['last_pr_merged_at']) else seq['last_pr_closed_at'],
            'last_category': seq['last_category'],
            'last_pr_state': seq['last_pr_state'],
            'last_merged_by': seq['last_merged_by'],
            'last_title': seq['last_title'],
            'metric': seq['metric'],
            'prior_superseded_prs': seq['prior_superseded_prs']
        })

# Convert the flattened result to a DataFrame
flattened_df = pd.DataFrame(flattened_result)
# flattened_df.dropna(subset=['last_category'], inplace=True)
flattened_df["repo_lib"] = flattened_df["repo"] + \
    "_" + flattened_df["lib_name"]

flattened_df.loc[:, 'pr_created_at'] = pd.to_datetime(
    flattened_df['pr_created_at'])
flattened_df.loc[:, 'last_pr_closed_at'] = pd.to_datetime(
    flattened_df['last_pr_closed_at'])
flattened_df.loc[:, 'last_pr_created_at'] = pd.to_datetime(
    flattened_df['last_pr_created_at'])
flattened_df.loc[:, 'last_pr_merged_at'] = pd.to_datetime(
    flattened_df['last_pr_merged_at'])

# flattened_df = flattened_df[flattened_df['state'] == "CLOSED"]

flattened_df.loc[:, 'delay_time'] = flattened_df.apply(calc_delay_time, axis=1)

flattened_df.dropna(subset=['delay_time'], inplace=True)

flattened_df['repo_pr_id'] = flattened_df['repo'].str.cat(flattened_df['id'].astype(str), "&SEP&")
flattened_df['repo_last_pr_id'] = flattened_df['repo'].str.cat(flattened_df['last_pr_id'].astype(str), "&SEP&")

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_58766/350054090.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby(['repo', 'lib_name']).apply(


In [29]:
all_upgrades_df = pd.read_csv('data/all_upgrades.csv')

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_58766/1630361064.py:1: DtypeWarning: Columns (25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  all_upgrades_df = pd.read_csv('data/all_upgrades.csv')


### Constructs upgrades' dataset

In [11]:

closed_prs_committedate_df = pd.read_csv("./data/closed_prs_committedate.csv")
# closed_prs_committedate_df = closed_prs_committedate_df[closed_prs_committedate_df['matching_commit_date_utc'].notna()]
closed_prs_committedate_df['repo_pr_id'] = closed_prs_committedate_df['repo'].str.cat(closed_prs_committedate_df['id'].astype(int).astype(str), '&SEP&')
closed_prs_committedate_df['matching_commit_date_utc'] = pd.to_datetime(closed_prs_committedate_df['matching_commit_date_utc'])

# closed_prs_committedate_kv = closed_prs_committedate_df.set_index('repo_pr_id')['matching_commit_date_utc'].to_dict()

cols = ['delay_time', 'title',  'dependabot_exists', 'id', 'repo', 'pr_created_at', 'last_pr_id',
        'last_pr_created_at', 'last_pr_merged_at', 'repo_pr_id', 'repo_last_pr_id', 'last_category', 'last_title', 'last_merged_by',
        'last_pr_state', 'continuous_superseded_count', 'prior_superseded_prs'
        ]
superseding_merged_PR_df = flattened_df.loc[(
    flattened_df['last_category'] == "Up-to-date") & (flattened_df['last_pr_state'] == "MERGED"), cols]
superseding_merged_PR_df['label'] = 'Superseding merged PR'

# def fill_closed_prs_last_pr_merged_at(row):
#     return closed_prs_committedate_kv.get(row['repo_pr_id'], row['last_pr_merged_at'])

superseding_closed_PR_df = flattened_df.loc[(
    flattened_df['last_category'] == "Up-to-date") & (flattened_df['last_pr_state'] == "CLOSED"), cols]
# superseding_closed_PR_loc_df = superseding_closed_PR_df.copy()

superseding_closed_prs_cols_to_assign = ['last_pr_state','last_pr_merged_at','last_pr_id','last_pr_closed_at', 'last_repo_pr_id', 'label']


reclassified_closed_superded_prs = pd.read_csv("data/supersede_results.csv")

superseding_closed_PR_Not_Null_df = superseding_closed_PR_df[superseding_closed_PR_df['repo_pr_id'].isin(reclassified_closed_superded_prs['repo_pr_id'].values)]
superseding_closed_PR_Not_Null_df.drop(columns=superseding_closed_prs_cols_to_assign, errors='ignore', inplace=True)

superseding_closed_prs_cols_to_assign += ['repo', 'id']

superseding_closed_PR_Not_Null_df = superseding_closed_PR_Not_Null_df.merge(
    reclassified_closed_superded_prs[superseding_closed_prs_cols_to_assign],
    on=['repo', 'id'],
    how='left'
)

superseding_closed_PR_df = pd.concat((superseding_closed_PR_df, superseding_closed_PR_Not_Null_df))
superseding_closed_PR_df.drop_duplicates(subset=['repo', 'id'], keep='last', inplace=True)


superseding_closed_PR_df['label'].fillna('Superseding closed PR with external upgrade', inplace=True)

superseding_closed_PR_df = pd.merge(
    superseding_closed_PR_df,
    closed_prs_committedate_df[['repo_pr_id', 'matching_commit_author_name', 'matching_commit_date_utc', 'matching_commit_hash']],
    on=['repo_pr_id'],
    how='left'
)

superseding_closed_PR_df['last_pr_merged_at'] = pd.to_datetime(superseding_closed_PR_df['last_pr_merged_at'])

superseding_closed_PR_df['delay_time'] = superseding_closed_PR_df.apply(lambda row: (
    row['last_pr_merged_at'] - row['pr_created_at']).total_seconds() / (60), axis=1)

# superseding_closed_PR_df['last_pr_merged_at'] = superseding_closed_PR_df.apply(fill_closed_prs_last_pr_merged_at, axis=1)

excluded_prs = superseding_merged_PR_df['repo_last_pr_id'].tolist(
) + superseding_closed_PR_df['repo_last_pr_id'].tolist()

closed_prs = df[(df['state'] == 'CLOSED') & (df['pr_category']
                                             == 'Up-to-date') & (~df['repo_pr_id'].isin(excluded_prs))]
closed_prs = closed_prs[cols[1:-11] + ['pr_closed_at', 'repo_pr_id']]
# closed_prs['label'] = 'Closed PR with external upgrade'

closed_prs_cols_to_assign = ['last_pr_state','last_pr_merged_at','last_pr_id','last_pr_closed_at', 'last_repo_pr_id', 'label']
closed_prs.drop(columns=closed_prs_cols_to_assign, errors='ignore', inplace=True)

closed_prs_cols_to_assign += ['repo', 'id']

closed_prs = closed_prs.merge(
    reclassified_closed_superded_prs[closed_prs_cols_to_assign],
    on=['repo', 'id'],
    how='left'
)

closed_prs = pd.merge(
    closed_prs,
    closed_prs_committedate_df[['repo_pr_id', 'matching_commit_author_name', 'matching_commit_date_utc', 'matching_commit_hash']],
    on=['repo_pr_id'],
    how='left'
)

# closed_prs['last_pr_merged_at'] = closed_prs.apply(fill_closed_prs_last_pr_merged_at, axis=1)

closed_prs['last_pr_merged_at'] = pd.to_datetime(closed_prs['last_pr_merged_at'])

closed_prs['delay_time'] = closed_prs.apply(lambda row: (
    row['last_pr_merged_at'] - row['pr_created_at']).total_seconds() / (60), axis=1)

excluded_prs += closed_prs.loc[closed_prs['last_pr_state']=='MERGED', 'last_repo_pr_id'].tolist() \
    + superseding_closed_PR_df.loc[superseding_closed_PR_df['last_pr_state']=='MERGED', 'last_repo_pr_id'].tolist()

merged_prs = df[(df['state'] == 'MERGED') & (
    ~df['repo_pr_id'].isin(excluded_prs))]
merged_prs['delay_time'] = merged_prs.apply(lambda row: (
    row['pr_merged_at'] - row['pr_created_at']).total_seconds() / (60), axis=1)
merged_prs = merged_prs[cols[:-11] + ['pr_merged_at']]
merged_prs['last_pr_merged_at'] = merged_prs['pr_merged_at']
merged_prs['label'] = 'Merged PR'

other_prs_df = pd.read_csv('data/closed_prs_committedate_others.csv')
other_prs_df = other_prs_df.dropna(subset=['matching_commit_author_name'])
other_prs_df['matching_commit_date_utc'] = pd.to_datetime(other_prs_df['matching_commit_date_utc'])

other_prs_cols = other_prs_df.columns.tolist()[-5:] + ['repo_pr_id']

others_superseded_prs_df = flattened_df.loc[flattened_df['repo_pr_id'].isin(other_prs_df['repo_pr_id'].values), cols]
others_superseded_prs_df =  pd.merge(
    others_superseded_prs_df,
    other_prs_df[other_prs_cols],
    on='repo_pr_id',
    how='left'
)
others_superseded_prs_df['label'] = 'Superseding Others'

others_closed_prs_ids = other_prs_df.loc[(other_prs_df['label']=='Closed Others'), 'repo_pr_id'].values
others_closed_prs_df = df.loc[df['repo_pr_id'].isin(others_closed_prs_ids), cols[1:-11] + ['pr_closed_at', 'repo_pr_id']]

others_closed_prs_df = pd.merge(
    others_closed_prs_df,
    other_prs_df[other_prs_cols],
    on=['repo_pr_id'],
    how='left'
)
others_closed_prs_df['label'] = 'Closed Others'


# Combine all upgrades of the 4 categories
all_upgrades_df = pd.concat(
    (merged_prs, closed_prs, superseding_merged_PR_df, superseding_closed_PR_df, others_superseded_prs_df, others_closed_prs_df))

# Sort the data by delay_time
all_upgrades_df = all_upgrades_df.sort_values(
    'delay_time').reset_index(drop=True)

# Calculate the indices for top and bottom 30%
n = len(all_upgrades_df)
bottom_index = int(0.3 * n)
top_index = n - int(0.3 * n)

# Create a new column for labeling
all_upgrades_df['delay_category'] = 'middle'
all_upgrades_df.loc[:bottom_index, 'delay_category'] = 'fast'
all_upgrades_df.loc[top_index:, 'delay_category'] = 'slow'

# all_upgrades_df = all_upgrades_df[(all_upgrades_df['delay_time'] >= 0)]

all_upgrades_df['repo_pr_id'] = all_upgrades_df['repo'].str.cat(
    all_upgrades_df['id'].astype(int).astype(str), '&SEP&')


def fill_missing_last_pr_merged_at(row):
    if pd.isna(row['last_pr_merged_at']):
        if pd.isna(row['pr_closed_at']):
            return row['pr_merged_at']
        else:
            return row['pr_closed_at']
    return row['last_pr_merged_at']


all_upgrades_df['last_pr_merged_at'] = all_upgrades_df.apply(
    fill_missing_last_pr_merged_at, axis=1)

all_upgrades_df = pd.merge(
    all_upgrades_df,
    df[['repo', 'id', 'merged_by']],
    on=['repo', 'id'],
    how='left'
)

all_upgrades_df['last_merged_by'] = all_upgrades_df.apply(
    lambda row: row['merged_by'] if pd.isna(row['last_merged_by']) else row['last_merged_by'], axis=1)

# retrieve the library repository from base_prs.csv file
## 1. Read base_prs.csv file
base_prs_df = pd.read_csv("../dependabot-security/data/base_prs.csv")

## 2. Extract the lib_repo in the base_prs file.
def extract_lib_repo(body: str) -> str:
    """
    Extract the library repository from the PR body text.

    Args:
        body (str): The body text of the PR.
    """
    lib_prov_name, _, _ = DependencyExtractor.extract_lib_provider_metadata(body)
    return lib_prov_name


base_prs_df['lib_repo'] = base_prs_df['body'].map(extract_lib_repo)

# 3. Merge with all_upgrades_df to get the unique libraries
all_upgrades_df = pd.merge(
    all_upgrades_df,
    base_prs_df[['repo', 'id', 'lib_repo']].drop_duplicates(),
    on=['repo', 'id'],
    how='left'
)

# 4. Assign TopiGPT labels
# Load jsonl into dataframe
topicgpt_data_df = pd.read_json("./data/output/sample/assignment_corrected.jsonl", lines=True)
topicgpt_data_df.rename(columns={'id': 'lib_repo', 'label': 'topicgpt_label'}, inplace=True)

def extract_label(response):
    import re
    if pd.isnull(response):
        return None
    match = re.search(r"\]\s*([^:]+):", response)
    if match:
        return match.group(1).strip()
    return None

topicgpt_data_df["topicgpt_label"] = topicgpt_data_df["responses"].apply(extract_label)

all_upgrades_df = pd.merge(
    all_upgrades_df,
    topicgpt_data_df[['lib_repo', 'topicgpt_label']].drop_duplicates(),
    on=['lib_repo'],
    how='left'
)

topic_label_alternatives = {
    "Serialization": "Data Serialization",
    "API": "Web Services",
    "HTTP": "Web Services",
    "Realtime communication": "WebSocket Libraries",
    "Geolocation": "Location Services",
    "Security": "Security Tools",
    "Database": "Database Tools",
    "Logging": "Logging Tools",
    "Release management": "Versioning",
    "Validation": "Data Validation",
    "Configuration": "Configuration Tools",
    "Internationalization": "i8n Libraries",
    "Frontend": "UI Libraries",
    "CLI": "CLI Libraries",
    "Runtime environment": "Utilities",
    "Filesystem": "Filesystem Tools",
    "Module loader": "Utilities",
    "Web framework": "Web Frameworks",
    "Testing": "Testing Tools"
}

all_upgrades_df['topicgpt_label'] = all_upgrades_df['topicgpt_label'].map(lambda x: topic_label_alternatives.get(x, x))


all_upgrades_df = pd.merge(
    all_upgrades_df,
    model_data_df[['repo', 'id', 'Dependency_Type_dev_runtime_runtime', 'Dependency_Type_dev_runtime_dev']],
    on=['repo', 'id'],
    how='left'
)

def refine_classification(row):
    label = row['label']
    author = row.get('matching_commit_author_name')
    commit_date = row.get('matching_commit_date_utc')
    commit_hash = row.get('matching_commit_hash')

    # Case 1: Already merged PR → no changes
    if label == 'Merged PR':
        return row

    # Case 2: Superseding merged PR → update merge date if available
    if label == 'Superseding merged PR':
        if pd.notna(author):
            row['last_pr_merged_at'] = commit_date
        return row

    # Case 3: Missing commit → exclude
    if pd.isna(commit_hash):
        row['label'] = f"To exclude"
        return row

    # Mapping logic for dependabot and others
    if label in ('Closed PR with external upgrade', 'Closed Others'):
        row['label'] = 'Merged PR' if author == 'dependabot[bot]' else 'Closed PR with external upgrade'
    elif label in ('Superseding closed PR with external upgrade', 'Superseding Others'):
        row['label'] = 'Superseding merged PR' if author == 'dependabot[bot]' else 'Superseding closed PR with external upgrade'

    # Update merge date
    row['last_pr_merged_at'] = commit_date
    return row

all_upgrades_df = all_upgrades_df.apply(refine_classification, axis=1)

all_upgrades_df = all_upgrades_df[all_upgrades_df['label'] != "To exclude"]

def retrieve_missing_superse_count(row):
    if row['label'] not in ['Superseding merged PR', 'Superseding closed PR with external upgrade'] or pd.notna(row['continuous_superseded_count']):
        return row['continuous_superseded_count']
    return 2

all_upgrades_df['continuous_superseded_count'] = all_upgrades_df.apply(retrieve_missing_superse_count, axis=1)

all_upgrades_df['delay_time'] = all_upgrades_df.apply(calc_delay_time, axis=1)
all_upgrades_df['delay_hours'] = all_upgrades_df['delay_time'] / 60
all_upgrades_df['delay_days'] = all_upgrades_df['delay_hours'] / 24

delayed_upgrades_df = all_upgrades_df[all_upgrades_df['delay_category'] == 'slow']

rapid_upgrades_df = all_upgrades_df[all_upgrades_df['delay_category'] == 'fast']

all_upgrades_df = all_upgrades_df[(all_upgrades_df['delay_time'] > 0)]

/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_46800/1505445881.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  superseding_closed_PR_Not_Null_df.drop(columns=superseding_closed_prs_cols_to_assign, errors='ignore', inplace=True)
/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_46800/1505445881.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  superseding_closed_PR_df = pd.concat((superseding_closed_PR_df, superseding_closed_PR_Not_Null_df))
/var/folders/p4/wn9d12jj6g78td0tl7gx7xlw0000gp/T/ipykernel_46800/1505445881.py:43: Futur

In [65]:
cols = ['repo','id','pr_created_at', 'repo_pr_id', 'title','pr_closed_at', 'repo_last_pr_id', 'last_pr_closed_at','last_pr_merged_at']
test_df = all_upgrades_df.loc[all_upgrades_df['label'].isin(['Closed PR with external upgrade', 'Superseding closed PR with external upgrade']), cols]
superseded_others = flattened_df.loc[flattened_df['last_category']=='Others', cols]
superseded_others['label'] = 'Superseding Others'
test_df = pd.concat((test_df, superseded_others))

existing_prs = test_df.loc[test_df['repo_pr_id'].notna(), 'repo_pr_id'].tolist() + test_df.loc[test_df['repo_last_pr_id'].notna(), 'repo_last_pr_id'].tolist()
closed_prs_others_df = df.loc[(df['pr_category']=='Others')&(~df['repo_pr_id'].isin(existing_prs)), cols[:6]]
closed_prs_others_df['label'] = 'Closed Others'
test_df = pd.concat((test_df, closed_prs_others_df))
test_df = test_df[test_df['label'].isin(['Superseding Others', 'Closed Others'])]

In [66]:
test_df = pd.merge(
    test_df,
    df[['repo_pr_id', 'changed_files', 'lib_name']],
    on='repo_pr_id',
    how='left'
)

In [67]:
test_df.to_csv("data/prs_to_classify_others.csv", index=None)

In [33]:
test_df = pd.merge(test_df, df[['repo', 'id', 'changed_files','lib_name']], on=['repo', 'id'], how='left')

In [35]:
test_df.to_csv("./data/prs_to_classify.csv", index=None)

In [38]:
selected_cols = ['repo', 'id', 'state', 'pr_created_at', 'pr_merged_at', 'pr_closed_at', 'merged_by', 'closed_by', 'repo_commit_count', 'repo_age', 'pull_request_count', 'pull_request_count', 'stargazer_count', 'mentionable_users_count']
df.loc[df['repo'].isin(all_upgrades_df['repo'].unique()), selected_cols].to_csv("data/clean_prs.csv", index=None)

In [3]:
from typing import Dict, List

SECURITY_LABEL = 'Security'
BUG_LABEL = 'Bug'
FEATURE_LABEL = 'Feature'
REFACTORING_LABEL = 'Refactoring'
DEPRECATE_LABEL = 'Deprecate'
TEST_LABEL = 'Test'
MERGE_LABEL = 'Merge'

def label_commit(commit_message: str) -> List[str]:
    """
    Label a commit based on its message according to the specified categories.

    Args:
        commit_message: The commit message to analyze

    Returns:
        List of labels applicable to this commit
    """
    message_lower = commit_message.lower()

    # Security labels (checked first as they might overlap with other categories)
    security_keywords = [
        # FROM this paper https://arxiv.org/pdf/2307.11853
        'attack', 'bypass', 'cve', 'dos', 'exploit', 'injection',
        'leakage', 'malicious', 'overflow', 'smuggling', 'unauthorized',
        'underflow', 'vulnerability', 'access control', 'open redirect', 'race condition',
        'denial of service', 'out of bound', 'dot dot slash',
        # From this paper https://arxiv.org/pdf/2105.14565
        'use after free', 'double free', 'divide by zero', 'illegal', 'disclosure',
        'improper', 'unexpected', 'sanity check', 'uninitialize', 'fail', 'null pointer dereference',
        'null function pointer', 'crash', 'corrupt', 'deadlock', 'fuzz', 'verify',
        'undefined behavior', 'exposure', 'remote code execution', 'osvdb', 'redos',
        'NVD', 'clickjack', 'man-in-the-middle', 'hijack', 'advisory', 'insecure', 'cross-origin',
        'infinite loop', 'authentication', 'brute force', 'crack', 'credential', 'hack', 'harden',
        'lockout', 'password', 'proof of concept', 'poison', 'privilege', 'spoof', 'compromise',
        'out of array', 'exhaust', 'off-by-one', 'privesc', 'bugzilla', 'constant time', 'mishandle',
        'underflow', 'violation', 'recursion', 'snprintf', 'guard', 'protect',

        #######
        'cross site', 'request forgery', 'csrf', 'xsrf', 'forged', 'security', 'vulnerable', 'backdoor',
        'threat', 'breach', 'violate', 'blacklist', 'overrun'
    ]
    if any(keyword in message_lower for keyword in security_keywords):
        return SECURITY_LABEL

    # Bug
    bug_keywords = ['fix', 'bug', 'repair', 'correct', 'prevent', 'issue',
                    'problem', 'error', 'exception', 'typo', 'failure']
    if any(keyword in message_lower for keyword in bug_keywords):
        return BUG_LABEL

    # Feature
    feature_phrases = ['add feature', 'new feature',
                       'create new', 'add new', 'add missing']
    feature_keywords = ['enable', 'add', 'update', 'improve', 'support',
                        'new', 'upgrade', 'optimize', 'implement']

    if any(phrase in message_lower for phrase in feature_phrases) or any(keyword in message_lower for keyword in feature_keywords):
        return FEATURE_LABEL

    # Test
    if 'test' in message_lower:
        return TEST_LABEL

    # Deprecate
    deprecate_keywords = ['deprecate', 'delete',
                          'remove', 'disable', 'obsolete', 'downgrade']
    if any(keyword in message_lower for keyword in deprecate_keywords):
        return DEPRECATE_LABEL

    # Refactoring
    refactor_keywords = ['refactor', 'refact', 'style']
    if any(keyword in message_lower for keyword in refactor_keywords):
        return REFACTORING_LABEL

    # Resource
    resource_keywords = ['config', 'licence',
                         'legal', 'readme', 'gitignore', 'doc']
    if any(keyword in message_lower for keyword in resource_keywords):
        return 'Resource'

    # Merge
    merge_keywords = ['merge', 'integrate']
    if (any(keyword in message_lower for keyword in merge_keywords) and
            not any(keyword in message_lower for keyword in ['fail', 'fix'])):
        return MERGE_LABEL

    # If no labels found, assign 'Others'
    # if not labels:
    return None


def label_commits(commits: str = None) -> Dict[str, List[str]]:
    """
    Label all commits in a list of (full_hash, short_hash, message) tuples.

    Args:
        commits: List of commit tuples (full_hash, short_hash, message)

    Returns:
        Dictionary mapping commit hashes to their labels
    """
    if pd.isna(commits):
        return None

    commits = commits.split("&SEP&")

    labels = []
    for commit in commits:
        label = label_commit(commit)
        if label:
            labels.append(label)
    # for label in labels:
    #     if SECURITY_LABEL in labels:
    #         return SECURITY_LABEL
    #     elif BUG_LABEL in labels:
    #         return BUG_LABEL
    #     elif FEATURE_LABEL in labels:
    #         return FEATURE_LABEL
    #     elif REFACTORING_LABEL in labels:
    #         return REFACTORING_LABEL
    #     elif DEPRECATE_LABEL in labels:
    #         return DEPRECATE_LABEL
    #     elif TEST_LABEL in labels:
    #         return TEST_LABEL
    #     elif MERGE_LABEL in labels:
    #         return MERGE_LABEL
    
    
            
    return labels


commits_between_versions_df = pd.read_csv("./data/commits_between_versions_all.csv")
commits_between_versions_df['repo_pr_id'] = commits_between_versions_df['repo'].str.cat(
    commits_between_versions_df['id'].astype(str), '&SEP&')

all_upgrades_df = pd.merge(
    all_upgrades_df,
    commits_between_versions_df[['repo_pr_id', 'commit_titles']],
    on=['repo_pr_id'],
    how='left'
)

all_upgrades_df['commit_label'] = all_upgrades_df['commit_titles'].apply(label_commits)

def fill_dependabot_exists(row):
    if pd.notna(row['dependabot_exists']):
        return row['dependabot_exists']
    
    res = pr_metrics_df[pr_metrics_df['repo_pr_id']==row['repo_pr_id']]
    
    if len(res) != 1:
        return row['dependabot_exists']
    return res['Dependabot_Config_File_Exist'].values[0]

all_upgrades_df['dependabot_exists'] = all_upgrades_df.apply(fill_dependabot_exists, axis=1)
all_upgrades_df['is_security'] = all_upgrades_df['dependabot_exists'].map(lambda x: "No" if x == True else "Yes")

In [15]:
def get_change_type(row):
    import re
    from packaging import version
    
    if pd.isna(row['last_title']):
        match = re.search(r'from ([0-9.]+) to ([0-9.]+)', row['title'])
        if not match:
            return None
        old_v, new_v = match.group(1), match.group(2)
    else:
        _, old_v, new_v1, _ = DependencyExtractor.extract(row['title'])
        _, _, new_v, _ = DependencyExtractor.extract(row['last_title'])
        
        if not new_v:
            new_v = new_v1
        
    old_v, new_v = version.parse(old_v), version.parse(new_v)

    if new_v.major > old_v.major:
        return "major"
    elif new_v.minor > old_v.minor:
        return "minor"
    elif new_v.micro > old_v.micro:
        return "patch"
    else:
        return None  # in case versions are the same

In [16]:
all_upgrades_df['change_type'] = all_upgrades_df.apply(get_change_type, axis=1)

In [274]:
# group counts per project and date
counts_per_project_day = (
    all_dfs.groupby(['repo', all_dfs['pr_created_at'].dt.date])
    .size()
    .rename('count')
    .reset_index()
    .rename(columns={'pr_created_at': 'date'})
)
counts_per_project_day['date'] = pd.to_datetime(counts_per_project_day['date'])

# function to get counts for a given PR


def get_total_counts(row):
    repo = row['repo']
    pr_date = row['pr_created_at'].date()

    # filter to same repo
    repo_counts = counts_per_project_day[counts_per_project_day['repo'] == repo]

    # same_day = repo_counts.loc[repo_counts['date'] == pd.to_datetime(pr_date), 'count'].sum()
    day_before = repo_counts.loc[(repo_counts['date'] >= pd.to_datetime(pr_date - pd.Timedelta(days=1))) & (
        repo_counts['date'] <= pd.to_datetime(pr_date + pd.Timedelta(days=1))), 'count'].sum()
    # day_after = repo_counts.loc[, 'count'].sum()

    # subtract 1 to exclude the delayed PR itself
    return max(0, day_before - 1)


def count_still_open(row):
    repo = row['repo']
    created_time = row['pr_created_at']

    repo_prs = df[df['repo'] == repo]

    # condition: created before this PR AND closed after this PR (or still open)
    still_open = repo_prs[
        (repo_prs['pr_created_at'] < created_time) &
        ((repo_prs['pr_closed_at'].isna()) |
         (repo_prs['pr_closed_at'] > created_time))
    ]

    return len(still_open)

In [275]:
delayed_upgrades_df['created_pr_before_upg_suggest_count'] = delayed_upgrades_df.apply(
    get_total_counts, axis=1)
delayed_upgrades_df['open_pr_before_upg_suggest_count'] = delayed_upgrades_df.apply(
    count_still_open, axis=1)

In [276]:
def count_prs_before_performing_upgrade(row):
    repo = row['repo']
    end_time = row['last_pr_merged_at']
    start_time = end_time - pd.Timedelta(days=1)

    # created PRs in same repo
    created_count = all_dfs[
        (all_dfs['repo'] == repo) &
        (all_dfs['pr_created_at'] <= end_time) &
        (all_dfs['pr_created_at'] >= start_time)
    ].shape[0]

    # open PRs in same repo
    open_count = all_dfs[
        (all_dfs['repo'] == repo) &
        (all_dfs['pr_created_at'] <= end_time) &
        (all_dfs['pr_closed_at'].isnull() |
         (all_dfs['pr_closed_at'] > end_time))
    ].shape[0]

    return pd.Series({
        'created_pr_before_perfor_upg_count': created_count,
        'open_pr_before_perfor_upg_count': open_count,
    })

# apply per delayed PR
delayed_upgrades_df = delayed_upgrades_df.join(
    delayed_upgrades_df.apply(count_prs_before_performing_upgrade, axis=1)
)

In [17]:
all_upgrades_df.to_csv("data/all_upgrades.csv", index=None)